# Anchor Honeypot: ISEA Classifier + Kamala Thinking Agent
End-to-end VS Code notebook for training/exporting the ISEA classifier and running a Kamala-only thinking layer on local Ollama.

In [1]:
%pip install -q -U torch transformers peft datasets accelerate scikit-learn pandas numpy matplotlib seaborn onnx onnxruntime optimum ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
assert sys.version_info >= (3, 10), f'Python 3.10+ required, got {sys.version}'
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


## Imports, Configuration, and Reproducibility

In [3]:
import os
import json
import time
import random
from dataclasses import dataclass, field
from typing import Dict, List, Any, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor, TimeoutError

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import ollama

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'microsoft/MiniLM-L12-H384-uncased'
MAX_LEN = 160
BATCH_SIZE = 16
EPOCHS = 2
SAVE_DIR = 'saved_models/isea_classifier'
ONNX_PATH = os.path.join(SAVE_DIR, 'model.onnx')
os.makedirs(SAVE_DIR, exist_ok=True)

print('Device:', DEVICE)

d:\Project-26\Anchor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [7]:
pip install ipywidgets widgetsnbextension

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.2 MB 6.3 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 4.6 MB/s  0:00:00
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------- ----- 786.4/914.9 kB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 3.5 MB/s  0:00:00

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Load ISEA Dataset and Normalize Schema

In [4]:
df = pd.read_csv('train.csv')
if 'attack_type' not in df.columns and 'class_label' in df.columns:
    df = df.rename(columns={'class_label': 'attack_type'})

df = df[['message_text', 'attack_type']].dropna().drop_duplicates()

canonical = {
    'benign': 'benign',
    'kyc_scam': 'KYC Scam',
    'impersonation': 'Impersonation',
    'phishing_link': 'Phishing Link',
    'fake_payment_portal': 'Fake Payment Portal',
    'account_block_scam': 'Account Block Scam'
}

def normalize_label(x: str) -> str:
    k = str(x).strip().lower().replace(' ', '_')
    if k in canonical:
        return canonical[k]
    for v in canonical.values():
        if str(x).strip().lower() == v.lower():
            return v
    raise ValueError(f'Unknown label: {x}')

df['attack_type'] = df['attack_type'].apply(normalize_label)

print(df.head(2))
print('\nClass distribution:')
print(df['attack_type'].value_counts())

                                        message_text attack_type
0  New water meter installation scheduled at your...      benign
1  Your water account 5774836158 has no pending d...      benign

Class distribution:
attack_type
benign                 2923
KYC Scam                929
Impersonation           909
Phishing Link           894
Fake Payment Portal     725
Account Block Scam      359
Name: count, dtype: int64


## Label Mapping, Split, and Tokenization

In [5]:
label_list = [
    'benign',
    'KYC Scam',
    'Impersonation',
    'Phishing Link',
    'Fake Payment Portal',
    'Account Block Scam'
]

label2id = {k: i for i, k in enumerate(label_list)}
id2label = {i: k for k, i in label2id.items()}
df['labels'] = df['attack_type'].map(label2id).astype(int)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['labels']
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(texts):
    return tokenizer(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN)

train_enc = tokenize_batch(train_df['message_text'])
val_enc = tokenize_batch(val_df['message_text'])

class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]))
        return item

train_ds = EncodedDataset(train_enc, train_df['labels'].tolist())
val_ds = EncodedDataset(val_enc, val_df['labels'].tolist())

print('Train size:', len(train_ds), 'Val size:', len(val_ds))

d:\Project-26\Anchor\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--microsoft--MiniLM-L12-H384-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Train size: 5391 Val size: 1348


## MiniLM + PEFT LoRA + Focal Loss

In [ ]:
# Cell A - Diagnostic (run this first)
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

num_labels = len(label_list)
assert num_labels == 6, f'Expected 6 labels, got {num_labels}'

print('Label mapping (id -> label):')
for i in range(num_labels):
    print(f'  {i}: {id2label[i]}')

train_unique = sorted(train_df['labels'].unique().tolist())
val_unique = sorted(val_df['labels'].unique().tolist())
print('Train unique labels:', train_unique)
print('Val unique labels:', val_unique)
assert train_unique == list(range(num_labels)), 'Train split missing classes'

sample = train_ds[0]
assert sample['labels'].dtype == torch.long
print('Dataset check passed. Label sample:', int(sample['labels']))

counts = train_df['labels'].value_counts().reindex(
    range(num_labels), fill_value=0).sort_index().values
beta = 0.999
effective_num = 1.0 - np.power(beta, counts)
alpha_np = (1.0 - beta) / np.clip(effective_num, 1e-12, None)
alpha_np = alpha_np / alpha_np.sum() * num_labels
alpha = torch.tensor(alpha_np, dtype=torch.float32)

print('\nClass weights:')
for i in range(num_labels):
    print(f'  {id2label[i]:>20}: count={int(counts[i]):4d}, weight={alpha[i].item():.4f}')

diag_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels,
    id2label=id2label, label2id=label2id
)
diag_lora = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8,
    lora_alpha=16, lora_dropout=0.1,
    target_modules=['query', 'value']
)
diag_model = get_peft_model(diag_model, diag_lora).to(DEVICE).eval()

batch = next(iter(DataLoader(train_ds, batch_size=4, shuffle=False)))
batch = {k: v.to(DEVICE) for k, v in batch.items()}
with torch.no_grad():
    out = diag_model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask']
    )

print('\nForward pass check:')
print('  logits shape:', tuple(out.logits.shape))
print('  predicted ids:', torch.argmax(out.logits, dim=-1).tolist())
assert out.logits.shape[1] == 6

del diag_model
print('\nDiagnostics passed. Ready to train.')


In [ ]:
# Cell B - Fixed training cell
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback, TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType

num_labels = len(label_list)
counts = train_df['labels'].value_counts().reindex(
    range(num_labels), fill_value=0).sort_index().values
beta = 0.999
effective_num = 1.0 - np.power(beta, counts)
alpha_np = (1.0 - beta) / np.clip(effective_num, 1e-12, None)
alpha_np = alpha_np / alpha_np.sum() * num_labels
alpha = torch.tensor(alpha_np, dtype=torch.float32)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(
            logits, targets, reduction='none')
        pt = torch.exp(-ce)
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]
        else:
            alpha_t = torch.ones_like(ce)
        return (alpha_t * ((1.0 - pt) ** self.gamma) * ce).mean()

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels,
    id2label=id2label, label2id=label2id
)
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=['query', 'value']
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    cm = confusion_matrix(labels, preds, labels=list(range(num_labels)))
    recalls = cm.diagonal() / np.clip(cm.sum(axis=1), 1, None)
    pred_counts = np.bincount(preds, minlength=num_labels)
    metrics = {
        'accuracy': float((preds == labels).mean()),
        'macro_recall': float(recalls.mean())
    }
    for i, name in enumerate(label_list):
        key = name.lower().replace(' ', '_')
        metrics[f'recall_{key}'] = float(recalls[i])
        metrics[f'pred_count_{key}'] = float(pred_counts[i])
    return metrics

class EpochPredictionLogger(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        epoch_no = int(round(state.epoch or 0))
        print(f'\nPer-class recall at epoch {epoch_no}:')
        for name in label_list:
            key = name.lower().replace(' ', '_')
            recall = float(metrics.get(f'eval_recall_{key}', 0.0))
            pred_count = int(metrics.get(f'eval_pred_count_{key}', 0))
            print(f'  {name:>20}: recall={recall:.4f}  pred_count={pred_count}')

class FocalTrainer(Trainer):
    def __init__(self, *args, focal_loss=None, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = focal_loss
        self.class_weights = class_weights

    def get_train_dataloader(self):
        labels_arr = np.array([
            int(self.train_dataset[i]['labels'].item())
            for i in range(len(self.train_dataset))
        ], dtype=np.int64)
        sample_weights = self.class_weights[labels_arr].cpu().double()
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=False
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = self.focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir='outputs/isea_lora_v2',
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='macro_recall',
    greater_is_better=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    group_by_length=True,
    seed=42
)

trainer = FocalTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    focal_loss=FocalLoss(alpha=alpha.to(DEVICE), gamma=1.5),
    class_weights=alpha.to(DEVICE),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2),
        EpochPredictionLogger()
    ]
)

print('Starting training with class-balanced sampling...')
trainer.train()
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best macro recall:', trainer.state.best_metric)

final_metrics = trainer.evaluate()
print('\nFinal evaluation:')
for k, v in final_metrics.items():
    if any(x in k for x in ['recall_', 'pred_count_', 'accuracy', 'macro_recall']):
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28426.76it/s]
BertForSequenceClassification LOAD REPORT from: microsoft/MiniLM-L12-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 149,766 || all params: 33,512,076 || trainable%: 0.4469


d:\Project-26\Anchor\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss


## Evaluation: Classification Report + Confusion Matrix

In [ ]:
pred = trainer.predict(val_ds)
y_true = val_df['labels'].values
y_pred = np.argmax(pred.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=label_list, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_list, yticklabels=label_list, cmap='Blues')
plt.title('ISEA Attack Classifier Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

def predict_attack_type(text: str) -> Tuple[str, float]:
    model_eval = trainer.model.to(DEVICE).eval()
    enc = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LEN)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        probs = torch.softmax(model_eval(**enc).logits, dim=-1)[0].detach().cpu().numpy()
    idx = int(np.argmax(probs))
    return id2label[idx], float(probs[idx])

## Save Model + Export ONNX

In [ ]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

export_model = trainer.model
if isinstance(export_model, PeftModel):
    export_model = export_model.merge_and_unload()
export_model = export_model.cpu().eval()

dummy = tokenizer('KYC update now', return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LEN)

torch.onnx.export(
    export_model,
    (dummy['input_ids'], dummy['attention_mask']),
    ONNX_PATH,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch', 1: 'seq'},
        'attention_mask': {0: 'batch', 1: 'seq'},
        'logits': {0: 'batch'}
    },
    opset_version=14
)

print('Saved to', SAVE_DIR)

import onnxruntime as ort
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
ort_logits = sess.run(None, {
    'input_ids': dummy['input_ids'].numpy(),
    'attention_mask': dummy['attention_mask'].numpy()
})[0]
with torch.no_grad():
    pt_logits = export_model(dummy['input_ids'], dummy['attention_mask']).logits.numpy()
print('Mean abs logit diff (PT vs ONNX):', np.mean(np.abs(pt_logits - ort_logits)))

## Ollama Connection Check

In [ ]:
import ollama
try:
    ollama.list()
    print('Ollama running')
except:
    print('Start Ollama first: ollama serve')
    print('Pull model: ollama pull llama3.2')

## Kamala System Prompt Builder

In [ ]:
KAMALA = {
    'name': 'Kamala',
    'age': 68,
    'language': 'Tamil',
    'trust_level': 0.72,
    'confusion_level': 0.62,
    'backstory': 'Retired school teacher from Coimbatore, Tamil Nadu.',
    'speech_patterns': ['short sentences', 'Tamil word mixing', 'asks for simple explanations'],
    'deflection_phrases': ['Konjam slow-aa sollunga', 'I will check and reply', 'I am confused, please repeat']
}

def build_kamala_system_prompt(scammer_intent: str, urgency_level: float, recent_turns: List[Dict[str, str]]) -> str:
    return f'''
You are Kamala, a 68-year-old retired school teacher from
Coimbatore, Tamil Nadu. You live alone and are not very
familiar with technology.

YOUR PERSONALITY:
- Warm, trusting, cooperative but genuinely confused by
  technical things
- You mix Tamil words naturally into conversation
- Short sentences because you type slowly
- Ask people to repeat or explain simply
- Get nervous when people mention bank account or Aadhaar
- Never make decisions quickly

YOUR RULES (never break these):
- Never give real personal information
- If asked for OTP say you didn't receive it or misread it
- If asked for Aadhaar give: 2318 5674 9023 (fake)
- If asked for account number give: 3901284756 (fake)
- If asked for UPI PIN say you forgot it
- Never sound like a robot
- Keep responses under 3 sentences

CURRENT SCAM CONTEXT: {scammer_intent}
URGENCY LEVEL: {urgency_level}
CONVERSATION SO FAR: {recent_turns}
'''.strip()

## Simplified ShortTermMemory

In [ ]:
@dataclass
class ShortTermMemory:
    session_id: str
    channel: str
    persona_name: str = 'Kamala'
    scammer_intent: str = 'unknown'
    bot_score: float = 0.0
    urgency_level: float = 0.0
    artifacts: List[Dict[str, Any]] = field(default_factory=list)
    deflections_used: List[str] = field(default_factory=list)
    recent_turns: List[Dict[str, str]] = field(default_factory=list)
    trust_level_given: float = 0.0
    jailbreak_attempts: int = 0

    def update_from_message(self, message: str):
        m = message.lower()
        if any(k in m for k in ['urgent', 'immediately', 'now', 'within']):
            self.urgency_level = min(1.0, self.urgency_level + 0.2)
        if any(k in m for k in ['ignore instructions', 'system prompt', 'developer mode']):
            self.jailbreak_attempts += 1
        self.recent_turns.append({'role': 'scammer', 'text': message})
        self.recent_turns = self.recent_turns[-3:]

    def add_artifact(self, artifact_type: str, value: str):
        self.artifacts.append({'type': artifact_type, 'value': value, 'ts': time.time()})

    def get_llm_context(self) -> Dict[str, Any]:
        return {
            'session_id': self.session_id,
            'channel': self.channel,
            'persona_name': self.persona_name,
            'scammer_intent': self.scammer_intent,
            'bot_score': round(self.bot_score, 4),
            'urgency_level': round(self.urgency_level, 4),
            'artifacts': self.artifacts[-5:],
            'deflections_used': self.deflections_used[-5:],
            'recent_turns': self.recent_turns[-3:],
            'trust_level_given': round(self.trust_level_given, 4),
            'jailbreak_attempts': self.jailbreak_attempts
        }

## Simplified ThinkingAgent (Kamala + Ollama llama3.2)

In [ ]:
class ThinkingAgent:
    def __init__(self, classifier_model, tokenizer, id2label, timeout_s: float = 0.8):
        self.classifier_model = classifier_model.to(DEVICE).eval()
        self.tokenizer = tokenizer
        self.id2label = id2label
        self.timeout_s = timeout_s

    def classify(self, text: str) -> Tuple[str, float]:
        enc = self.tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LEN)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            probs = torch.softmax(self.classifier_model(**enc).logits, dim=-1)[0].cpu().numpy()
        idx = int(np.argmax(probs))
        return self.id2label[idx], float(probs[idx])

    def bot_score(self, text: str) -> float:
        features = ['http://', 'https://', 'hxxp', 'tinyurl', 'bit.ly', 'urgent', 'verify', 'kyc', 'suspended', 'blocked']
        t = text.lower()
        return min(1.0, sum(int(f in t) for f in features) / 10.0)

    def _fallback(self) -> str:
        return 'Ayyo, I am little confused. Konjam simple-aa sollunga, I will check and reply.'

    def _ollama_chat(self, system_prompt: str, user_message: str, turns: List[Dict[str, str]]) -> str:
        transcript = '\n'.join([f"{x['role']}: {x['text']}" for x in turns[-3:]])
        prompt = f"Recent turns:\n{transcript}\n\nLatest scammer message:\n{user_message}"

        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': prompt}
        ]

        def invoke():
            resp = ollama.chat(
                model='llama3.2',
                messages=messages,
                options={'temperature': 0.7},
                keep_alive='30s'
            )
            return resp.get('message', {}).get('content', '').strip()

        with ThreadPoolExecutor(max_workers=1) as ex:
            fut = ex.submit(invoke)
            return fut.result(timeout=self.timeout_s)

    def process(self, raw_message: str, stm: ShortTermMemory) -> Optional[str]:
        attack_type, _ = self.classify(raw_message)
        stm.scammer_intent = attack_type
        stm.update_from_message(raw_message)

        stm.bot_score = self.bot_score(raw_message)
        if stm.bot_score >= 0.65:
            return None

        system_prompt = build_kamala_system_prompt(
            scammer_intent=stm.scammer_intent,
            urgency_level=stm.urgency_level,
            recent_turns=stm.recent_turns[-3:]
        )

        try:
            response = self._ollama_chat(system_prompt, raw_message, stm.recent_turns)
            if not response:
                response = self._fallback()
        except (TimeoutError, Exception):
            response = self._fallback()

        stm.recent_turns.append({'role': 'anchor', 'text': response})
        stm.recent_turns = stm.recent_turns[-3:]
        stm.deflections_used.append('Konjam slow-aa sollunga')
        stm.trust_level_given = KAMALA['trust_level']
        return response

## Full Pipeline Test (5 Messages)

In [ ]:
agent = ThinkingAgent(trainer.model if 'trainer' in globals() else model, tokenizer, id2label, timeout_s=0.8)
stm = ShortTermMemory(session_id='sess-001', channel='message')

samples = [
    'KYC expired. Update immediately at https://tiny.one/abc',
    'Hi this is your bank support team. Share customer ID for verification.',
    'Power cut in 1 hour due to dues. Pay at http://pay-portal-now.example',
    'UPI refund pending. Claim now at https://bit.ly/refund',
    'Your account blocked due to suspicious activity. Verify now.'
]

rows = []
for i, m in enumerate(samples, start=1):
    pred, conf = agent.classify(m)
    reply = agent.process(m, stm)
    rows.append({
        'turn': i,
        'predicted_attack_type': pred,
        'confidence': round(conf, 4),
        'persona_selected': 'Kamala',
        'llm_response': reply,
        'bot_score': round(stm.bot_score, 4)
    })

    print('\n' + '=' * 90)
    print(f'Turn {i}')
    print('Classifier output:', pred, f'({conf:.4f})')
    print('Persona selected:', 'Kamala')
    print('LLM response:', reply)
    print('ShortTermMemory state:')
    print(json.dumps(stm.get_llm_context(), indent=2, ensure_ascii=True))

display(pd.DataFrame(rows))